# Tugas Data Wrangling Kesehatan

Notebook ini menjalankan Task 1–5 secara reproducible dari `Dataset_Kesehatan_Real_Names.xlsx`. Semua persentase pada dataset disimpan sebagai proporsi (misalnya 0,30 = 30%).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from wrangling_pipeline import (
    add_stunting_category,
    filter_stunting_and_poverty_2023,
    find_iqr_outliers,
    health_correlation,
    make_long_data,
    make_stunting_panel,
    merge_dimension_and_facts,
    minmax_by_island,
    split_tables,
    summarize_health_by_region,
)

INPUT = Path('Dataset_Kesehatan_Real_Names.xlsx')
data = pd.read_excel(INPUT)
print(f'Ukuran data awal: {data.shape}')
print(f'Duplikasi baris: {data.duplicated().sum()}')
display(data.isna().sum().rename('Jumlah missing').to_frame().T)

## Task 1 — Pemeriksaan dan pemisahan tabel

Data dipisahkan menjadi dimensi wilayah yang unik dan tabel fakta kesehatan agar atribut wilayah tidak berulang pada setiap indikator.

In [ ]:
dimension, facts = split_tables(data)
print('Dimensi wilayah:', dimension.shape)
print('Tabel fakta:', facts.shape)
display(dimension.head())
display(facts.head())

**Interpretasi.** Tabel dimensi berisi 514 kabupaten/kota unik, sedangkan tabel fakta mempertahankan 2.570 observasi (514 wilayah × 5 tahun).

## Task 2 — Data long format dan pivot stunting

Dua puluh tiga indikator diubah ke bentuk tidy/long. Panel stunting dipivot agar perbandingan antar-tahun per kabupaten/kota mudah dibaca.

In [ ]:
long_data = make_long_data(data)
long_data.to_csv('dataset long format.csv', index=False)
stunting_panel = make_stunting_panel(data)
print('Ukuran long data:', long_data.shape)
print('Kolom long data:', list(long_data.columns))
display(stunting_panel.head())

**Interpretasi.** Long data memiliki 59.110 baris (= 2.570 observasi × 23 indikator) dan empat kolom yang diminta. Pivot menampilkan tahun 2015, 2018, 2019, 2021, dan 2023.

## Task 3 — Filter risiko dan left join

Wilayah berisiko dipilih pada tahun 2023 saat prevalensi stunting > 0,30 dan penduduk miskin > 0,10. Left join memakai tabel dimensi sebagai sisi kiri.

In [ ]:
risk_2023 = filter_stunting_and_poverty_2023(data)
joined = merge_dimension_and_facts(dimension, facts)
print('Wilayah risiko 2023:', len(risk_2023))
print('Ukuran hasil left join:', joined.shape)
display(risk_2023[['Kabupaten/Kota', 'Provinsi', 'Prevalensi Stunting (%)', 'Penduduk Miskin (%)']].head())

**Interpretasi.** Filter hanya mengambil observasi 2023 yang melewati kedua ambang. Left join menghasilkan tepat 2.570 baris sehingga tidak ada observasi fakta yang hilang atau tergandakan.

## Task 4 — Agregasi hierarkis, normalisasi, dan kategori

AKI dan stunting diringkas dengan mean serta simpangan baku menurut Wilayah Indonesia, Pulau, dan Tipe Wilayah. Kepadatan serta PDRB dinormalisasi min–max di dalam setiap pulau.

In [ ]:
hierarchical_summary = summarize_health_by_region(data)
normalized_2023 = minmax_by_island(data)
categorized = add_stunting_category(data)
category_counts = (categorized.loc[categorized['Tahun'].eq(2023), 'Kategori Stunting']
                   .value_counts()
                   .reindex(['Rendah', 'Sedang', 'Tinggi / Kritis'], fill_value=0))
display(hierarchical_summary.head(12))
display(normalized_2023[['Kabupaten/Kota', 'Pulau', 'Kepadatan Penduduk Normal', 'PDRB per Kapita Normal']].head())
display(category_counts.rename('Jumlah wilayah').to_frame())

**Interpretasi.** Mean menunjukkan tingkat rata-rata indikator pada tiap hirarki, sedangkan simpangan baku menunjukkan ketimpangannya. Nilai normalisasi 0–1 hanya membandingkan wilayah dalam pulau yang sama. Untuk 2023: <0,20 = Rendah; 0,20–0,30 = Sedang; >0,30 = Tinggi / Kritis.

## Task 5 — Outlier, korelasi, dan visualisasi

Outlier tahun 2023 dideteksi terpisah untuk PDRB dan kepadatan dengan pagar 1,5 × IQR. Korelasi menggunakan Pearson untuk lima indikator kesehatan/sosial-ekonomi.

In [ ]:
outliers_2023 = find_iqr_outliers(data)
correlation = health_correlation(data)
print('Jumlah outlier per variabel:')
display(outliers_2023.groupby('Variabel').size().rename('Jumlah').to_frame())
display(correlation.round(3))

In [ ]:
# Heatmap korelasi matplotlib (fallback karena seaborn tidak tersedia)
fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(correlation.to_numpy(), cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(correlation.columns)), correlation.columns, rotation=35, ha='right')
ax.set_yticks(range(len(correlation.index)), correlation.index)
for row in range(len(correlation.index)):
    for column in range(len(correlation.columns)):
        ax.text(column, row, f'{correlation.iloc[row, column]:.2f}', ha='center', va='center', fontsize=9)
fig.colorbar(image, ax=ax, label='Korelasi Pearson')
ax.set_title('Heatmap Korelasi Pearson Indikator Kesehatan')
fig.tight_layout()
fig.savefig('heatmap_korelasi.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplot stunting 2023 menurut pulau
stunting_2023 = data.loc[data['Tahun'].eq(2023)]
islands = sorted(stunting_2023['Pulau'].dropna().unique())
box_values = [stunting_2023.loc[stunting_2023['Pulau'].eq(island), 'Prevalensi Stunting (%)'].dropna() for island in islands]
fig, ax = plt.subplots(figsize=(11, 6))
ax.boxplot(box_values, tick_labels=islands)
ax.set_title('Distribusi Prevalensi Stunting menurut Pulau (2023)')
ax.set_xlabel('Pulau')
ax.set_ylabel('Prevalensi Stunting')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig('boxplot_stunting_pulau.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Tren rata-rata stunting menurut Wilayah Indonesia
regional_trend = (data.groupby(['Tahun', 'Wilayah Indonesia'], as_index=False)['Prevalensi Stunting (%)']
                 .mean())
fig, ax = plt.subplots(figsize=(10, 6))
for region, values in regional_trend.groupby('Wilayah Indonesia'):
    ax.plot(values['Tahun'], values['Prevalensi Stunting (%)'], marker='o', label=region)
ax.set_title('Tren Rata-rata Stunting menurut Wilayah Indonesia')
ax.set_xlabel('Tahun')
ax.set_ylabel('Rata-rata Prevalensi Stunting')
ax.set_xticks(sorted(data['Tahun'].unique()))
ax.grid(alpha=0.25)
ax.legend(title='Wilayah Indonesia', bbox_to_anchor=(1.02, 1), loc='upper left')
fig.tight_layout()
fig.savefig('tren_stunting_wilayah.png', dpi=160, bbox_inches='tight')
plt.show()

**Interpretasi.** Outlier menandai kabupaten/kota dengan nilai PDRB atau kepadatan yang sangat jauh dari sebaran nasional 2023. Heatmap menunjukkan arah/kekuatan hubungan linear, boxplot memperlihatkan variasi antar-pulau, dan grafik garis menunjukkan perubahan rata-rata stunting per wilayah dari waktu ke waktu.

## Kesimpulan

Dataset berhasil dinormalisasi ke bentuk long, dianalisis secara hierarkis, dan divisualisasikan secara reproducible. Analisis 2023 memisahkan wilayah berisiko tinggi, mengelompokkan tingkat stunting, serta mengidentifikasi nilai ekonomi/demografi yang ekstrem. Hasil ini dapat dipakai sebagai dasar prioritisasi intervensi kesehatan dengan tetap mempertimbangkan perbedaan antar-pulau dan tipe wilayah.